In [5]:
import pandas as pd

# Load datasets
weather_data = pd.read_csv('Weather report (2019 -2024).csv', skiprows=1)
forecast_data = pd.read_csv('Forcast Weather.csv')

# Standardize column names
weather_data.columns = [
    "Date", "Max Temp (°F)", "Avg Temp (°F)", "Min Temp (°F)",
    "Max Dew Point (°F)", "Avg Dew Point (°F)", "Min Dew Point (°F)",
    "Max Humidity (%)", "Avg Humidity (%)", "Min Humidity (%)",
    "Max Wind Speed (mph)", "Avg Wind Speed (mph)", "Min Wind Speed (mph)",
    "Max Pressure (in)", "Avg Pressure (in)", "Min Pressure (in)",
    "Precipitation (in)"
]

# Fill missing values and ensure 'Date' is in datetime format
weather_data.fillna(0, inplace=True)
weather_data['Date'] = pd.to_datetime(weather_data['Date'], format="%m/%d/%y", errors='coerce')
weather_data = weather_data[weather_data['Date'].notna()]

# Convert numeric columns to proper types
numeric_columns = [
    "Max Temp (°F)", "Avg Temp (°F)", "Min Temp (°F)",
    "Max Dew Point (°F)", "Avg Dew Point (°F)", "Min Dew Point (°F)",
    "Max Humidity (%)", "Avg Humidity (%)", "Min Humidity (%)",
    "Max Wind Speed (mph)", "Avg Wind Speed (mph)", "Min Wind Speed (mph)",
    "Max Pressure (in)", "Avg Pressure (in)", "Min Pressure (in)",
    "Precipitation (in)"
]
weather_data[numeric_columns] = weather_data[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Add Year, Month, and Day columns
weather_data['Year'] = weather_data['Date'].dt.year
weather_data['Month'] = weather_data['Date'].dt.month_name()
weather_data['Day'] = weather_data['Date'].dt.day

forecast_data.columns = [
    "Date", "Max Temp (°F)", "Avg Temp (°F)", "Min Temp (°F)",
    "Max Dew Point (°F)", "Avg Dew Point (°F)", "Min Dew Point (°F)",
    "Max Humidity (%)", "Avg Humidity (%)", "Min Humidity (%)",
    "Max Wind Speed (mph)", "Avg Wind Speed (mph)", "Min Wind Speed (mph)",
    "Max Pressure (in)", "Avg Pressure (in)", "Min Pressure (in)",
    "Precipitation (in)"
]
forecast_data.fillna(0, inplace=True)
forecast_data['Date'] = pd.to_datetime(forecast_data['Date'], errors='coerce')
forecast_data = forecast_data[forecast_data['Date'].notna()]
forecast_data = forecast_data[forecast_data['Date'] > pd.Timestamp.now().normalize()]

# Function to get valid menu input from the user
def get_valid_menu_input(prompt, min_val, max_val):
    while True:
        try:
            value = int(input(prompt))
            if min_val <= value <= max_val:
                return value
            else:
                print(f"Invalid Entry. Please enter a number between {min_val} and {max_val}.")
        except ValueError:
            print("Invalid Entry. Please enter a valid number.")

# Menu options
def display_current_weather():
    current_date = pd.Timestamp.now().normalize()
    current_data = weather_data.loc[weather_data['Date'] == current_date]
    if current_data.empty:
        print("No current weather data available for today. Showing the latest available data:")
        latest_data = weather_data.iloc[-1]
        print(f"Date: {latest_data['Date'].strftime('%A, %Y-%m-%d')}")
        print(f"Temperature: {latest_data['Avg Temp (°F)']}°F")
        print(f"Humidity: {latest_data['Avg Humidity (%)']}%")
        print(f"Wind Speed: {latest_data['Avg Wind Speed (mph)']} mph")
    else:
        current = current_data.iloc[0]
        print(f"Date: {current['Date'].strftime('%A, %Y-%m-%d')}")
        print(f"Temperature: {current['Avg Temp (°F)']}°F")
        print(f"Humidity: {current['Avg Humidity (%)']}%")
        print(f"Wind Speed: {current['Avg Wind Speed (mph)']} mph")

def display_monthly_weather():
    last_month = pd.Timestamp.now() - pd.DateOffset(months=1)
    monthly_data = weather_data[
        (weather_data['Year'] == last_month.year) &
        (weather_data['Month'] == last_month.strftime('%B'))
    ]
    if monthly_data.empty:
        print("No data for the last month.")
    else:
        avg_temp = pd.to_numeric(monthly_data['Avg Temp (°F)'], errors='coerce').mean()
        avg_humidity = pd.to_numeric(monthly_data['Avg Humidity (%)'], errors='coerce').mean()
        avg_wind_speed = pd.to_numeric(monthly_data['Avg Wind Speed (mph)'], errors='coerce').mean()
        print(f"\nAverage Weather Data for {last_month.strftime('%B').upper()}/{last_month.year}")
        print(f"Average Temperature: {avg_temp:.2f}°F")
        print(f"Average Humidity: {avg_humidity:.2f}%")
        print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

def display_historical_weather():
    try:
        start_date = pd.to_datetime(input("Enter the start date (YYYY-MM-DD): "))
        end_date = pd.to_datetime(input("Enter the end date (YYYY-MM-DD): "))
    except ValueError:
        print("Invalid date format. Please enter dates in the format YYYY-MM-DD.")
        return

    if (end_date - start_date).days > 7:
        print("Error: Maximum date range is 7 days.")
        return

    historical_data = weather_data[
        (weather_data['Date'] >= start_date) & (weather_data['Date'] <= end_date)
    ]
    if historical_data.empty:
        print("No historical data available for the given range.")
    else:
        avg_temp = pd.to_numeric(historical_data['Avg Temp (°F)'], errors='coerce').mean()
        avg_humidity = pd.to_numeric(historical_data['Avg Humidity (%)'], errors='coerce').mean()
        avg_wind_speed = pd.to_numeric(historical_data['Avg Wind Speed (mph)'], errors='coerce').mean()
        print(f"\nHistorical Weather Data from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
        print(f"Average Temperature: {avg_temp:.2f}°F")
        print(f"Average Humidity: {avg_humidity:.2f}%")
        print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

def display_forecast_weather():
    try:
        forecast_date = pd.to_datetime(input("Enter the forecast date (YYYY-MM-DD): "))
    except ValueError:
        print("Invalid date format. Please enter dates in the format YYYY-MM-DD.")
        return

    if forecast_date <= pd.Timestamp.now().normalize():
        print("Error: Please enter a date after today's date for the forecast.")
        return

    forecast_data_for_date = forecast_data[forecast_data['Date'] == forecast_date]
    if forecast_data_for_date.empty:
        print("No forecast data available for the given date.")
    else:
        avg_temp = pd.to_numeric(forecast_data_for_date['Avg Temp (°F)'], errors='coerce').mean()
        avg_humidity = pd.to_numeric(forecast_data_for_date['Avg Humidity (%)'], errors='coerce').mean()
        avg_wind_speed = pd.to_numeric(forecast_data_for_date['Avg Wind Speed (mph)'], errors='coerce').mean()
        print(f"\nForecast Weather Data for {forecast_date.strftime('%A, %Y-%m-%d')}")
        print(f"Average Temperature: {avg_temp:.2f}°F")
        print(f"Average Humidity: {avg_humidity:.2f}%")
        print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

# Main menu function to display options and link to weather dashboard
def main_menu():
    while True:
        print("\nWeather Dashboard Main Menu")
        print("1. Current Weather")
        print("2. Monthly Weather")
        print("3. Historical Weather")
        print("4. Forecast Weather")
        print("5. Exit")

        menu_input = get_valid_menu_input("Please enter menu selection (1 - 5): ", 1, 5)

        if menu_input == 1:
            display_current_weather()
        elif menu_input == 2:
            display_monthly_weather()
        elif menu_input == 3:
            display_historical_weather()
        elif menu_input == 4:
            display_forecast_weather()
        elif menu_input == 5:
            print("Exiting the Weather Dashboard.")
            break

# Run the main menu
if __name__ == "__main__":
    main_menu()




Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  1


Date: Wednesday, 2024-11-27
Temperature: 48.1°F
Humidity: 31.0%
Wind Speed: 5.5 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  2



Average Weather Data for OCTOBER/2024
Average Temperature: 76.53°F
Average Humidity: 20.65%
Average Wind Speed: 7.16 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


...


Historical Weather Data from 2024-10-25 to 2024-10-27
Average Temperature: 70.90°F
Average Humidity: 25.83%
Average Wind Speed: 3.90 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  4
Enter the forecast date (YYYY-MM-DD):  2024-11-28


No forecast data available for the given date.

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  5


Exiting the Weather Dashboard.


In [1]:
import json, requests, pprint
import pandas as pd

# please enter your appid here
APPID = '3741775b9523fb5654ad764de205b071'
# assume the location to be las vegas. You can also ask the user to enter the location
# The format of the location should be "cityName, 2-letter_country_code"
location = 'Las Vegas, NV, US'

# Function to get weather data from OpenWeatherMap API
def get_weather_data(endpoint, params):
    url = f'https://api.openweathermap.org/data/2.5/{endpoint}'
    params['q'] = location
    params['appid'] = APPID
    params['units'] = 'Imperial'
    response = requests.get(url, params=params)
    response.raise_for_status()
    return json.loads(response.text)

# Function to display current weather
def display_current_weather():
    weather_data = get_weather_data('weather', {})
    print('Current weather in', weather_data['name'])
    weather_sec = weather_data['weather'][0]['description']
    main_sec = weather_data['main']
    print(f"Description: {weather_sec}")
    print(f"Temperature: {main_sec['temp']}°F")
    print(f"Feels like: {main_sec['feels_like']}°F")
    print(f"Humidity: {main_sec['humidity']}%")

# Function to display forecast weather
def display_forecast_weather():
    forecast_data = get_weather_data('forecast', {'cnt': 40})  # 5-day forecast
    forecast_date = input("Enter the forecast date (YYYY-MM-DD): ")
    forecast_for_date = [f for f in forecast_data['list'] if f['dt_txt'].startswith(forecast_date)]
    if not forecast_for_date:
        print("No forecast data available for the given date.")
    else:
        avg_temp = sum(f['main']['temp'] for f in forecast_for_date) / len(forecast_for_date)
        avg_humidity = sum(f['main']['humidity'] for f in forecast_for_date) / len(forecast_for_date)
        avg_wind_speed = sum(f['wind']['speed'] for f in forecast_for_date) / len(forecast_for_date)
        print(f"\nForecast Weather Data for {forecast_date}")
        print(f"Average Temperature: {avg_temp:.2f}°F")
        print(f"Average Humidity: {avg_humidity:.2f}%")
        print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

# Function to display historical weather
def display_historical_weather():
    print("Historical data is not directly available from the free OpenWeatherMap API.")
    # Note: Historical data requires a paid subscription and is typically fetched via different endpoints.

# Function to display monthly weather (5-day forecast data as a rough estimate)
def display_monthly_weather():
    forecast_data = get_weather_data('forecast', {'cnt': 40})  # 5-day forecast
    avg_temp = sum(f['main']['temp'] for f in forecast_data['list']) / len(forecast_data['list'])
    avg_humidity = sum(f['main']['humidity'] for f in forecast_data['list']) / len(forecast_data['list'])
    avg_wind_speed = sum(f['wind']['speed'] for f in forecast_data['list']) / len(forecast_data['list'])
    print(f"\nAverage Monthly Weather Estimate (Based on 5-Day Forecast)")
    print(f"Average Temperature: {avg_temp:.2f}°F")
    print(f"Average Humidity: {avg_humidity:.2f}%")
    print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

# Main menu function to display options and link to weather dashboard
def main_menu():
    while True:
        print("\nWeather Dashboard Main Menu")
        print("1. Current Weather")
        print("2. Monthly Weather")
        print("3. Historical Weather")
        print("4. Forecast Weather")
        print("5. Exit")

        menu_input = input("Please enter menu selection (1 - 5): ")

        if menu_input == "1":
            display_current_weather()
        elif menu_input == "2":
            display_monthly_weather()
        elif menu_input == "3":
            display_historical_weather()
        elif menu_input == "4":
            display_forecast_weather()
        elif menu_input == "5":
            print("Exiting the Weather Dashboard.")
            break
        else:
            print("Invalid choice. Please try again.")

# Run the main menu
if __name__ == "__main__":
    main_menu()



Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  1


Current weather in Las Vegas
Description: overcast clouds
Temperature: 59.04°F
Feels like: 56.14°F
Humidity: 32%

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  2



Average Monthly Weather Estimate (Based on 5-Day Forecast)
Average Temperature: 50.21°F
Average Humidity: 20.70%
Average Wind Speed: 4.12 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


...

Historical data is not directly available from the free OpenWeatherMap API.

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  4
Enter the forecast date (YYYY-MM-DD):  2024-11-28



Forecast Weather Data for 2024-11-28
Average Temperature: 48.75°F
Average Humidity: 28.38%
Average Wind Speed: 6.57 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  5


Exiting the Weather Dashboard.


In [7]:
import json, requests, pprint
import pandas as pd

# please enter your appid here
APPID = '3741775b9523fb5654ad764de205b071'
# assume the location to be las vegas. You can also ask the user to enter the location
# The format of the location should be "cityName, 2-letter_country_code"
location = 'Las Vegas, NV, US'

# Load historical weather data
weather_data = pd.read_csv('Weather report (2019 -2024).csv', skiprows=1)

# Standardize column names
weather_data.columns = [
    "Date", "Max Temp (\u00b0F)", "Avg Temp (\u00b0F)", "Min Temp (\u00b0F)",
    "Max Dew Point (\u00b0F)", "Avg Dew Point (\u00b0F)", "Min Dew Point (\u00b0F)",
    "Max Humidity (%)", "Avg Humidity (%)", "Min Humidity (%)",
    "Max Wind Speed (mph)", "Avg Wind Speed (mph)", "Min Wind Speed (mph)",
    "Max Pressure (in)", "Avg Pressure (in)", "Min Pressure (in)",
    "Precipitation (in)"
]

# Fill missing values and ensure 'Date' is in datetime format
weather_data.fillna(0, inplace=True)
weather_data['Date'] = pd.to_datetime(weather_data['Date'], format="%m/%d/%y", errors='coerce')
weather_data = weather_data[weather_data['Date'].notna()]

# Convert numeric columns to proper types
numeric_columns = [
    "Max Temp (\u00b0F)", "Avg Temp (\u00b0F)", "Min Temp (\u00b0F)",
    "Max Dew Point (\u00b0F)", "Avg Dew Point (\u00b0F)", "Min Dew Point (\u00b0F)",
    "Max Humidity (%)", "Avg Humidity (%)", "Min Humidity (%)",
    "Max Wind Speed (mph)", "Avg Wind Speed (mph)", "Min Wind Speed (mph)",
    "Max Pressure (in)", "Avg Pressure (in)", "Min Pressure (in)",
    "Precipitation (in)"
]
weather_data[numeric_columns] = weather_data[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Function to get weather data from OpenWeatherMap API
def get_weather_data(endpoint, params):
    url = f'https://api.openweathermap.org/data/2.5/{endpoint}'
    params['q'] = location
    params['appid'] = APPID
    params['units'] = 'Imperial'
    response = requests.get(url, params=params)
    response.raise_for_status()
    return json.loads(response.text)

# Function to display current weather
def display_current_weather():
    weather_data = get_weather_data('weather', {})
    print('Current weather in', weather_data['name'])
    weather_sec = weather_data['weather'][0]['description']
    main_sec = weather_data['main']
    print(f"Description: {weather_sec}")
    print(f"Temperature: {main_sec['temp']}\u00b0F")
    print(f"Feels like: {main_sec['feels_like']}\u00b0F")
    print(f"Humidity: {main_sec['humidity']}%")

# Function to display forecast weather
def display_forecast_weather():
    forecast_data = get_weather_data('forecast', {'cnt': 40})  # 5-day forecast
    forecast_date = input("Enter the forecast date (YYYY-MM-DD): ")
    forecast_for_date = [f for f in forecast_data['list'] if f['dt_txt'].startswith(forecast_date)]
    if not forecast_for_date:
        print("No forecast data available for the given date.")
    else:
        avg_temp = sum(f['main']['temp'] for f in forecast_for_date) / len(forecast_for_date)
        avg_humidity = sum(f['main']['humidity'] for f in forecast_for_date) / len(forecast_for_date)
        avg_wind_speed = sum(f['wind']['speed'] for f in forecast_for_date) / len(forecast_for_date)
        print(f"\nForecast Weather Data for {forecast_date}")
        print(f"Average Temperature: {avg_temp:.2f}\u00b0F")
        print(f"Average Humidity: {avg_humidity:.2f}%")
        print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

# Function to display historical weather
def display_historical_weather():
    try:
        start_date = pd.to_datetime(input("Enter the start date (YYYY-MM-DD): "))
        end_date = pd.to_datetime(input("Enter the end date (YYYY-MM-DD): "))
    except ValueError:
        print("Invalid date format. Please enter dates in the format YYYY-MM-DD.")
        return

    if (end_date - start_date).days > 7:
        print("Error: Maximum date range is 7 days.")
        return

    historical_data = weather_data[
        (weather_data['Date'] >= start_date) & (weather_data['Date'] <= end_date)
    ]
    if historical_data.empty:
        print("No historical data available for the given range.")
    else:
        avg_temp = pd.to_numeric(historical_data['Avg Temp (\u00b0F)'], errors='coerce').mean()
        avg_humidity = pd.to_numeric(historical_data['Avg Humidity (%)'], errors='coerce').mean()
        avg_wind_speed = pd.to_numeric(historical_data['Avg Wind Speed (mph)'], errors='coerce').mean()
        print(f"\nHistorical Weather Data from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
        print(f"Average Temperature: {avg_temp:.2f}\u00b0F")
        print(f"Average Humidity: {avg_humidity:.2f}%")
        print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

# Function to display monthly weather (5-day forecast data as a rough estimate)
def display_monthly_weather():
    forecast_data = get_weather_data('forecast', {'cnt': 40})  # 5-day forecast
    avg_temp = sum(f['main']['temp'] for f in forecast_data['list']) / len(forecast_data['list'])
    avg_humidity = sum(f['main']['humidity'] for f in forecast_data['list']) / len(forecast_data['list'])
    avg_wind_speed = sum(f['wind']['speed'] for f in forecast_data['list']) / len(forecast_data['list'])
    print(f"\nAverage Monthly Weather Estimate (Based on 5-Day Forecast)")
    print(f"Average Temperature: {avg_temp:.2f}\u00b0F")
    print(f"Average Humidity: {avg_humidity:.2f}%")
    print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

# Main menu function to display options and link to weather dashboard
def main_menu():
    while True:
        print("\nWeather Dashboard Main Menu")
        print("1. Current Weather")
        print("2. Monthly Weather")
        print("3. Historical Weather")
        print("4. Forecast Weather")
        print("5. Exit")

        menu_input = input("Please enter menu selection (1 - 5): ")

        if menu_input == "1":
            display_current_weather()
        elif menu_input == "2":
            display_monthly_weather()
        elif menu_input == "3":
            display_historical_weather()
        elif menu_input == "4":
            display_forecast_weather()
        elif menu_input == "5":
            print("Exiting the Weather Dashboard.")
            break
        else:
            print("Invalid choice. Please try again.")

# Run the main menu
if __name__ == "__main__":
    main_menu()



Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  1


Current weather in Las Vegas
Description: few clouds
Temperature: 55.74°F
Feels like: 52.61°F
Humidity: 34%

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  2



Average Monthly Weather Estimate (Based on 5-Day Forecast)
Average Temperature: 50.53°F
Average Humidity: 19.12%
Average Wind Speed: 3.69 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


...

Invalid choice. Please try again.

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  4
Enter the forecast date (YYYY-MM-DD):  2024-11-28



Forecast Weather Data for 2024-11-28
Average Temperature: 49.36°F
Average Humidity: 32.43%
Average Wind Speed: 6.71 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  5


Exiting the Weather Dashboard.
